# 01 — Ingesta y armonización de microdatos EPH

Lee los 86 trimestres publicados cacheados en `data/eph_cache/` (2003T3–2025T4, en
3 formatos distintos: DBF histórico, `.txt` regular con 2+ convenciones de nombre),
normaliza **estructura** (nombres de columna, tipos de las claves de vinculación) sin
tocar ningún valor, y deja:

- Parquet particionado por `ANO4`/`TRIMESTRE` en `data/01_armonizado/` (gitignored, se
  regenera corriendo este notebook).
- Un inventario de variables (presencia y tipo real por trimestre) para saber qué se
  puede comparar entre trimestres y qué no.
- Una tabla de flags metodológicos por trimestre en `data/meta/flags_trimestre.csv`
  (versionada).

**Parámetros** (celda de setup): `TRIMESTRES` (subconjunto opcional, `None` = todos) y
`REHACER` (`False` por defecto -- si la partición ya existe en `01_armonizado/`, no vuelve
a leer `eph_cache/`, relee el Parquet ya escrito para los agregados cross-trimestre). Una
corrida con `TRIMESTRES` acotado no pisa `inventario_variables.csv`/`flags_trimestre.csv`
globales -- los escribe con sufijo `_parcial`.

No se recodifica ningún valor especial (`-9`, `9`, etc.) — solo se documentan.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd


def _raiz_repo(desde: Path | None = None) -> Path:
    """Sube desde `desde` (por defecto el cwd del kernel) buscando `pytest.ini`
    como marca de la raíz del repo."""
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

from eph_client import EphClient, TrimestreNoPublicado, UrlDesconocida
import armonizacion as arm
import pipeline_meta as pm

TRIMESTRES: list[tuple[int, int]] | None = None  # subconjunto opcional; None = todos los publicados
REHACER: bool = False  # True fuerza reprocesar aunque la partición ya exista en 01_armonizado

client = EphClient(cache_dir=REPO / "data" / "eph_cache")
ARMONIZADO = REPO / "data" / "01_armonizado"
META = REPO / "data" / "meta"
ARMONIZADO.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)

trimestres = TRIMESTRES if TRIMESTRES is not None else arm.trimestres_publicados()
ES_CORRIDA_PARCIAL = TRIMESTRES is not None
COMMIT_GIT = pm.obtener_commit_git(REPO)
print(f"{len(trimestres)} trimestres a procesar" +
      (" -- CORRIDA PARCIAL (TRIMESTRES acotado)" if ES_CORRIDA_PARCIAL else " (2003T3-2025T4, sin los no publicados)"))

## 1. Lectura + armonización + inventario (un solo pase)

El "tipo" de una variable puede cambiar entre trimestres (verificado: `IPCF` es
`float64` en toda la serie salvo en 2021T1 y 2021T3, donde queda `str` -- el mismo
trimestre con punto decimal en vez de coma que ya documenta
`tipado_nucleo.detectar_formato_decimal`, no un caso de "entero a float") — así
que el inventario tiene que salir de la lectura real, no de una muestra. Como ya hay que
leer cada trimestre completo para escribirlo a Parquet, se hace todo en el mismo pase:
por trimestre y base, se lee, se arma, se registra en el inventario, y se escribe — sin
guardar los 86 DataFrames en memoria a la vez.

Los trimestres `.rar` sin `unar` (`armonizacion.pendiente_rar`) se saltean acá y quedan
marcados en la tabla de flags, no rompen la corrida.

In [ ]:
registros_inventario = []
aglomerados_por_trimestre = {}
saltados = []
reusados = []
escritos = []

t0 = time.time()
for anio, trimestre in trimestres:
    filas_resumen_trimestre = []
    for tipo in ("hogar", "individual"):
        destino_path = pm.ruta_particion(ARMONIZADO, tipo, anio, trimestre)
        ya_existe = destino_path.exists()

        if ya_existe and not REHACER:
            # No re-arma desde data/eph_cache/ (se saltea el parseo caro) -- pero
            # sí relee lo ya escrito para alimentar inventario/aglomerados igual
            # que si se hubiera reprocesado (ver plan de reorganización, punto 4).
            df_arm = pd.read_parquet(destino_path)
            reusados.append({"anio": anio, "trimestre": trimestre, "tipo": tipo, "filas": len(df_arm)})
            filas_resumen_trimestre.append({
                "anio": anio, "trimestre": trimestre, "tipo": tipo,
                "filas_escritas": len(df_arm), "saltado": True, "motivo_saltado": "ya existía",
            })
        else:
            try:
                df = arm.leer_trimestre(client, anio, trimestre, tipo)
            except (TrimestreNoPublicado, UrlDesconocida, RuntimeError) as e:
                saltados.append({"anio": anio, "trimestre": trimestre, "tipo": tipo, "motivo": str(e)})
                filas_resumen_trimestre.append({
                    "anio": anio, "trimestre": trimestre, "tipo": tipo,
                    "filas_escritas": 0, "saltado": True, "motivo_saltado": str(e)[:120],
                })
                continue

            df_arm = arm.armonizar(df, tipo)
            destino_path.parent.mkdir(parents=True, exist_ok=True)
            df_arm.to_parquet(destino_path, index=False)
            escritos.append({"anio": anio, "trimestre": trimestre, "tipo": tipo, "filas": len(df_arm)})
            filas_resumen_trimestre.append({
                "anio": anio, "trimestre": trimestre, "tipo": tipo,
                "filas_escritas": len(df_arm), "saltado": False, "motivo_saltado": "",
            })

        registros_inventario.extend(arm.registrar_inventario(df_arm, anio, trimestre, tipo))
        if tipo == "individual":
            aglomerados_por_trimestre[(anio, trimestre)] = set(
                df_arm["AGLOMERADO"].dropna().astype(int).tolist()
            )

    pm.actualizar_resumen_trimestre(META / "resumen_01_armonizado.csv", filas_resumen_trimestre)

duracion_total = time.time() - t0
print(f"tiempo total: {duracion_total:.1f}s")
print(f"escritos: {len(escritos)} -- reusados (ya existían): {len(reusados)} -- saltados: {len(saltados)}")
for s in saltados:
    print("  saltado:", s["anio"], s["trimestre"], s["tipo"], "--", s["motivo"][:80])

## 2. Inventario de variables

Presencia y tipo real por trimestre, sobre todos los trimestres efectivamente leídos.

In [ ]:
inventario = arm.construir_inventario(registros_inventario)
print(f"{inventario.shape[0]} filas (variable x trimestre x tipo)")
for tipo in ("individual", "hogar"):
    n_variables = inventario[inventario["tipo"] == tipo]["variable"].nunique()
    n_trimestres = inventario[inventario["tipo"] == tipo][["anio", "trimestre"]].drop_duplicates().shape[0]
    print(f"{tipo}: {n_variables} variables distintas vistas alguna vez, {n_trimestres} trimestres verificados")

In [ ]:
sufijo = "_parcial" if ES_CORRIDA_PARCIAL else ""
destino_inventario = META / f"inventario_variables{sufijo}.csv"
inventario.to_csv(destino_inventario, index=False)
print(f"guardado: {destino_inventario} ({len(inventario)} filas)" +
      (" -- CORRIDA PARCIAL, no pisa el inventario global" if ES_CORRIDA_PARCIAL else ""))

Cantidad de columnas por trimestre, para visualizar el salto de 2023T4 sin adivinar.

In [ ]:
conteo_columnas = (
    inventario.groupby(["tipo", "anio", "trimestre"])["variable"]
    .nunique()
    .reset_index(name="n_variables")
    .sort_values(["tipo", "anio", "trimestre"])
)
conteo_columnas[conteo_columnas["tipo"] == "individual"].tail(10)

In [ ]:
huecos = arm.variables_con_hueco(inventario)
print(f"{len(huecos)} variables con huecos (presentes en algunos trimestres, ausentes en otros)")
huecos.sort_values("trimestres_presente").head(15)

In [ ]:
inconsistentes = arm.tipos_inconsistentes(inventario)
print(f"{len(inconsistentes)} variables con más de un tipo de dato observado entre trimestres")
inconsistentes.sort_values("cantidad_tipos", ascending=False).head(15)

Heurística de nombre -- NO es detección semántica, ver docstring de `posibles_renombres`.
Sirve como punto de partida para revisión humana, no como verdad ya confirmada.

In [ ]:
renombres = arm.posibles_renombres(inventario)
print(f"{len(renombres)} pares (variable_anterior, variable_posible_reemplazo) candidatos")
renombres

## 3. Variables núcleo

Presencia y tipo por trimestre para las variables que probablemente use cualquier
indicador de deterioro de hogares, más el bloque de vivienda (`IV*`/`II*` de la base
hogar, tomado dinámicamente del inventario, no de una lista fija a mano). Se marca
explícitamente si una variable cambia de lado (tipo o presencia) justo en el corte
2023T3→2023T4.

In [ ]:
NUCLEO_HOGAR = ["IPCF", "ITF", "PONDIH"]
NUCLEO_INDIVIDUAL = ["P21", "T_VI", "PONDERA", "ESTADO", "CAT_OCUP", "PP07H", "CH04", "CH06"]

vivienda = sorted(
    inventario[(inventario["tipo"] == "hogar") & inventario["variable"].str.match(r"^(IV|II)")][
        "variable"
    ].unique()
)
print(f"{len(vivienda)} variables de vivienda detectadas dinámicamente: {vivienda}")

variables_nucleo = (
    [("hogar", v) for v in NUCLEO_HOGAR + vivienda]
    + [("individual", v) for v in NUCLEO_INDIVIDUAL]
)

In [ ]:
CORTE_2023T4 = (2023, 4)
CORTE_ANTERIOR = (2023, 3)

filas_nucleo = []
for tipo, variable in variables_nucleo:
    sub = inventario[(inventario["tipo"] == tipo) & (inventario["variable"] == variable)]
    presente_antes = not sub[(sub["anio"] == CORTE_ANTERIOR[0]) & (sub["trimestre"] == CORTE_ANTERIOR[1])].empty
    fila_despues = sub[(sub["anio"] == CORTE_2023T4[0]) & (sub["trimestre"] == CORTE_2023T4[1])]
    presente_despues = not fila_despues.empty
    tipo_antes = sub[(sub["anio"] == CORTE_ANTERIOR[0]) & (sub["trimestre"] == CORTE_ANTERIOR[1])]["tipo_dato"]
    tipo_antes = tipo_antes.iloc[0] if not tipo_antes.empty else None
    tipo_despues = fila_despues["tipo_dato"].iloc[0] if presente_despues else None
    afectada = (presente_antes != presente_despues) or (presente_antes and presente_despues and tipo_antes != tipo_despues)
    filas_nucleo.append({
        "tipo": tipo,
        "variable": variable,
        "trimestres_presente": sub[["anio", "trimestre"]].drop_duplicates().shape[0],
        "tipos_observados": sorted(sub["tipo_dato"].unique()),
        "afectada_2023t4": bool(afectada),
    })

tabla_nucleo = pd.DataFrame(filas_nucleo)
tabla_nucleo

**Nota sobre el bloque de vivienda (D2)**: la tabla de arriba marca la mayoría
de las variables `II*`/`IV*` con `afectada_2023t4=True` (cambia de tipo entre
2023T3 y 2023T4), pero esto es artefacto de NA -- 2023T3 tiene ~0.1% de
faltantes que fuerzan `float64` -- no una redefinición real de códigos. Los
códigos del bloque de vivienda no cambian entre eras (confirmado contra el
codebook, ver D2 en `docs/decisiones_metodologicas.md`).

## 4. Flags por trimestre

`aglomerados_faltantes` se calcula contra el conjunto "esperado" de cada era (pre/post
2006T3), no está hardcodeado -- si aparece algún trimestre con hueco que no se había
verificado antes a mano, este cálculo lo encuentra igual.

In [ ]:
esperados = arm.aglomerados_esperados(aglomerados_por_trimestre)
print("aglomerados esperados por era:", {k: sorted(v) for k, v in esperados.items()})

registros_flags = []
for anio, trimestre in trimestres:
    codigos = aglomerados_por_trimestre.get((anio, trimestre))
    faltantes = arm.aglomerados_faltantes(codigos, anio, trimestre, esperados) if codigos is not None else []
    registros_flags.append(arm.flags_trimestre(anio, trimestre, faltantes))

flags = arm.tabla_flags(registros_flags)
flags[flags["aglomerados_faltantes"].apply(len) > 0]

In [ ]:
sufijo = "_parcial" if ES_CORRIDA_PARCIAL else ""
flags_csv = flags.assign(
    aglomerados_faltantes=lambda d: d["aglomerados_faltantes"].apply(lambda l: ",".join(map(str, l)))
)
destino_flags = META / f"flags_trimestre{sufijo}.csv"
flags_csv.to_csv(destino_flags, index=False)
print(f"guardado: {destino_flags} ({len(flags_csv)} filas)" +
      (" -- CORRIDA PARCIAL, no pisa flags_trimestre.csv global" if ES_CORRIDA_PARCIAL else ""))
flags[flags["pendiente_rar"]]

## 5. Validación con DuckDB

Se lee el Parquet particionado con `union_by_name=true` porque los trimestres no
comparten el mismo conjunto de columnas (177 vs. 235 en individual, por el quiebre de
2023T4) -- sin esa opción DuckDB alinearía columnas por posición, no por nombre, y
mezclaría datos de columnas distintas.

In [ ]:
import duckdb

con = duckdb.connect()
ind_glob = str(ARMONIZADO / "individual" / "**" / "*.parquet")
hog_glob = str(ARMONIZADO / "hogar" / "**" / "*.parquet")

# CAST explícito a SMALLINT (= Int16): el archivo Parquet ya guarda ANO4/TRIMESTRE
# como int16 (verificado con pyarrow.parquet.read_schema), pero el lector de
# read_parquet de DuckDB 1.5.5 los ensancha a BIGINT al leerlos -- se lo revierte acá
# explícitamente en la vista, como corresponde según lo verificado más abajo.
con.sql(f"""
    CREATE OR REPLACE VIEW individual AS
    SELECT * REPLACE (CAST(ANO4 AS SMALLINT) AS ANO4, CAST(TRIMESTRE AS SMALLINT) AS TRIMESTRE)
    FROM read_parquet('{ind_glob}', union_by_name=true)
""")
con.sql(f"""
    CREATE OR REPLACE VIEW hogar AS
    SELECT * REPLACE (CAST(ANO4 AS SMALLINT) AS ANO4, CAST(TRIMESTRE AS SMALLINT) AS TRIMESTRE)
    FROM read_parquet('{hog_glob}', union_by_name=true)
""")

print("filas totales individual:", con.sql("SELECT COUNT(*) FROM individual").fetchone()[0])
print("filas totales hogar:", con.sql("SELECT COUNT(*) FROM hogar").fetchone()[0])

Verificación en dos partes: (1) el Parquet en disco quedó en `int16` tal como lo
escribió pandas/pyarrow; (2) DuckDB 1.5.5 ensancha `int16` a `BIGINT` al leer con
`read_parquet` -- por eso la vista de la celda anterior ya lo castea explícitamente
a `SMALLINT`, y acá se confirma que con el cast queda `SMALLINT` como se pidió.

In [ ]:
import pyarrow.parquet as pq

un_archivo = next((ARMONIZADO / "individual").rglob("*.parquet"))
schema_archivo = pq.read_schema(un_archivo)
tipos_archivo = {campo.name: str(campo.type) for campo in schema_archivo if campo.name in ("ANO4", "TRIMESTRE")}
print("tipo declarado en el archivo Parquet (pyarrow):", tipos_archivo)
assert all(t == "int16" for t in tipos_archivo.values()), "el Parquet no quedó en int16"

tipos_releidos = con.sql(
    "SELECT column_name, column_type FROM (DESCRIBE SELECT ANO4, TRIMESTRE FROM individual)"
).df()
print(tipos_releidos)
assert set(tipos_releidos["column_type"]) == {"SMALLINT"}, "la vista no quedó en SMALLINT tras el cast"
print("Parquet en disco: int16 (OK, sin necesitar corrección).")
print("Vista SQL de DuckDB: SMALLINT tras cast explícito (el lector de DuckDB ensancha a BIGINT sin el cast).")

Filas y viviendas (`CODUSU`) distintas por trimestre, ambas bases.

In [ ]:
resumen = con.sql("""
    SELECT 'individual' AS base, ANO4, TRIMESTRE, COUNT(*) AS filas, COUNT(DISTINCT CODUSU) AS codusu_distintos
    FROM individual GROUP BY ANO4, TRIMESTRE
    UNION ALL
    SELECT 'hogar' AS base, ANO4, TRIMESTRE, COUNT(*) AS filas, COUNT(DISTINCT CODUSU) AS codusu_distintos
    FROM hogar GROUP BY ANO4, TRIMESTRE
    ORDER BY ANO4, TRIMESTRE, base
""").df()
resumen.tail(10)

Duplicados de clave dentro de cada trimestre.

In [ ]:
dup_hogar = con.sql("""
    SELECT ANO4, TRIMESTRE, COUNT(*) AS filas, COUNT(DISTINCT CODUSU || '-' || NRO_HOGAR) AS claves_unicas
    FROM hogar GROUP BY ANO4, TRIMESTRE
    HAVING filas <> claves_unicas
""").df()
dup_individual = con.sql("""
    SELECT ANO4, TRIMESTRE, COUNT(*) AS filas,
           COUNT(DISTINCT CODUSU || '-' || NRO_HOGAR || '-' || COMPONENTE) AS claves_unicas
    FROM individual GROUP BY ANO4, TRIMESTRE
    HAVING filas <> claves_unicas
""").df()
print(f"trimestres con clave duplicada en hogar: {len(dup_hogar)}")
print(f"trimestres con clave duplicada en individual: {len(dup_individual)}")
dup_hogar, dup_individual

Hogares sin ninguna persona asociada, y personas sin hogar asociado.

In [ ]:
hogares_huerfanos = con.sql("""
    SELECT h.ANO4, h.TRIMESTRE, COUNT(*) AS hogares_sin_persona
    FROM hogar h
    ANTI JOIN individual i
      ON h.CODUSU = i.CODUSU AND h.NRO_HOGAR = i.NRO_HOGAR
     AND h.ANO4 = i.ANO4 AND h.TRIMESTRE = i.TRIMESTRE
    GROUP BY h.ANO4, h.TRIMESTRE
""").df()
personas_huerfanas = con.sql("""
    SELECT i.ANO4, i.TRIMESTRE, COUNT(*) AS personas_sin_hogar
    FROM individual i
    ANTI JOIN hogar h
      ON h.CODUSU = i.CODUSU AND h.NRO_HOGAR = i.NRO_HOGAR
     AND h.ANO4 = i.ANO4 AND h.TRIMESTRE = i.TRIMESTRE
    GROUP BY i.ANO4, i.TRIMESTRE
""").df()
print(f"trimestres con hogares sin persona: {len(hogares_huerfanos)}")
print(f"trimestres con personas sin hogar: {len(personas_huerfanas)}")
hogares_huerfanos, personas_huerfanas

## 6. Resumen final

No se redacta acá la entrada del log de decisiones sobre el quiebre de 2023T4 (D2) --
se propone fuera del notebook, con el usuario, usando los resultados reales de la
sección 3 (variables núcleo afectadas) en vez de una muestra manual.

In [ ]:
n_filas_totales = sum(e["filas"] for e in escritos)
pm.registrar_corrida(
    META / "corridas.csv",
    notebook="01_ingesta_armonizacion",
    parametros={"TRIMESTRES": TRIMESTRES, "REHACER": REHACER},
    n_trimestres=len(trimestres),
    n_filas=n_filas_totales,
    duracion_segundos=duracion_total,
    commit_git=COMMIT_GIT,
)

print("Trimestres escritos (nuevo):", len({(e['anio'], e['trimestre']) for e in escritos}))
print("Trimestres reusados (ya existían):", len({(r['anio'], r['trimestre']) for r in reusados}))
print("Trimestres saltados (sin unar u otro motivo):", len({(s['anio'], s['trimestre']) for s in saltados}))
print("Variables núcleo afectadas por el quiebre 2023T4:")
print(tabla_nucleo[tabla_nucleo["afectada_2023t4"]][["tipo", "variable"]].to_string(index=False))